In [ ]:
import pandas as pd
import numpy as np
import unicodedata
import random

# =============================================================================
# SCRIPT 1: SIMULACIÓN DE HECHOS SEMANALES CON MOTOR DE TEXTO (NLP READY)
# =============================================================================
np.random.seed(42)
random.seed(42)

def clean_text(text):
    return unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('utf-8')

estructuras = [
    ("1. Ingenieria de Detalle", ["1.1 Topografia y Suelos", "1.2 Diseno Civil", "1.3 Diseno Electromecanico", "1.4 Estudios Especiales"], ["Porticos 115kV", "Caseta de Control", "Trafo de Potencia", "Vias Internas", "Drenajes", "Malla a Tierra", "SSAA", "Comunicaciones", "Cerramiento", "Iluminacion"]),
    ("2. Gestion Administrativa", ["2.1 Licencias Ambientales", "2.2 Permisos Municipales", "2.3 Gestion Predial", "2.4 Polizas y Seguros"], ["Fase 1", "Fase 2", "Fase 3", "Lote Principal", "Zonas de Acceso", "Torres de Linea", "Campamento", "Bodegas", "Maquinaria", "Personal"]),
    ("3. Obras Civiles", ["3.1 Movimiento de Tierras", "3.2 Cimentaciones Mayores", "3.3 Cimentaciones Menores", "3.4 Estructuras de Concreto"], ["Bases de Trafo", "Porticos Llegada", "Porticos Salida", "Carcamos MT", "Carcamos BT", "Muros Cortafuego", "Placa Caseta", "Vias Perimetrales", "Drenajes Lluvias", "Bases Equipos"]),
    ("4. Equipos Primarios", ["4.1 Suministro Transformador", "4.2 Suministro Maniobra", "4.3 Suministro Medida", "4.4 Suministro Proteccion"], ["Unidad Principal", "Reserva", "Interruptor 115kV", "Seccionador 115kV", "CTs 115kV", "PTs 115kV", "Pararrayos", "Celdas 13.8kV", "Banco Condensadores", "Aisladores Soporte"]),
    ("5. Montaje Electromecanico", ["5.1 Estructuras Metalicas", "5.2 Equipos de Patio", "5.3 Barras y Conexiones", "5.4 Apriete y Nivelacion"], ["Porticos 115kV", "Soportes Equipos", "Trafo Principal", "Interruptores", "Seccionadores", "CTs y PTs", "Tuberia Aluminio", "Cable ACSR", "Herrajes", "Grapas"]),
    ("6. Control y Protecciones", ["6.1 Tableros de Linea", "6.2 Tableros de Trafo", "6.3 Servicios Auxiliares", "6.4 Cableado de Control"], ["Gabinete 1", "Gabinete 2", "Gabinete 3", "Banco Baterias", "Cargador", "Tendido Cable", "Conexionado", "Fibra Optica", "Switch Ethernet", "RTU"]),
    ("7. Pruebas y Puesta", ["7.1 Pruebas Equipos", "7.2 Pruebas Control", "7.3 Pruebas de Sistema", "7.4 Energizacion"], ["Transformador", "Interruptores", "Seccionadores", "Reles Distancia", "Reles Diferenciales", "Integracion SCADA", "Medidores", "Lazos de Control", "Protocolos SAT", "Acta Final"])
]

actividades_base = []
for cap_idx, (cap_name, subcapitulos, elementos) in enumerate(estructuras):
    for sub_idx, sub_name in enumerate(subcapitulos):
        for elem_idx, elemento in enumerate(elementos):
            wbs_id = f"{cap_idx+1}.{sub_idx+1}.{elem_idx+1}"
            actividad = f"{sub_name.split(' ', 1)[1]} - {elemento}"
            bac = round(np.random.uniform(5000, 25000), 2)

            if cap_idx == 0:   s_start, s_end = 1, 24
            elif cap_idx == 1: s_start, s_end = 1, 60
            elif cap_idx == 2: s_start, s_end = 5, 32
            elif cap_idx == 3: s_start, s_end = 8, 40
            elif cap_idx == 4: s_start, s_end = 28, 50
            elif cap_idx == 5: s_start, s_end = 36, 56
            elif cap_idx == 6: s_start, s_end = 48, 60

            if 20 < s_start: pv_20 = 0.0
            elif 20 >= s_end: pv_20 = bac
            else: pv_20 = bac * (((20 - s_start) / (s_end - s_start)) ** 1.2)

            if cap_idx == 0:   ev_20 = pv_20 * np.clip(np.random.normal(0.98, 0.02), 0.9, 1.0)
            elif cap_idx == 1: ev_20 = pv_20 * np.clip(np.random.normal(0.95, 0.02), 0.9, 1.0)
            elif cap_idx == 2: ev_20 = pv_20 * np.clip(np.random.normal(0.60, 0.1), 0.4, 0.8)
            elif cap_idx == 3: ev_20 = pv_20 * np.clip(np.random.normal(0.70, 0.1), 0.5, 0.8)
            else:              ev_20 = 0.0

            actividades_base.append({
                'WBS': wbs_id, 'Capitulo': clean_text(cap_name), 'Subcapitulo': clean_text(sub_name),
                'Actividad': clean_text(actividad), 'BAC': bac, 's_start': s_start, 's_end': s_end,
                'pv_20': pv_20, 'ev_20': ev_20, 'ac_20': 0.0
            })

df_base = pd.DataFrame(actividades_base)

# Absorción matemática para CPI = 0.82
Target_AC = df_base['ev_20'].sum() / 0.82
m1, m2 = df_base['Capitulo'].str.startswith('1.'), df_base['Capitulo'].str.startswith('2.')
m3, m4 = df_base['Capitulo'].str.startswith('3.'), df_base['Capitulo'].str.startswith('4.')

df_base.loc[m1, 'ac_20'] = df_base.loc[m1, 'ev_20'] / 0.98
df_base.loc[m2, 'ac_20'] = df_base.loc[m2, 'ev_20'] / 0.95
df_base.loc[m4, 'ac_20'] = df_base.loc[m4, 'ev_20'] / 0.85
AC_used = df_base.loc[m1 | m2 | m4, 'ac_20'].sum()
df_base.loc[m3, 'ac_20'] = (df_base.loc[m3, 'ev_20'] / df_base.loc[m3, 'ev_20'].sum()) * (Target_AC - AC_used)

# =============================================================================
# DICCIONARIOS DE CAMPO (Generación de Sentimiento y Causa Raíz)
# =============================================================================
log_civ_ok = ["Inicio de excavacion en terreno seco.", "Avance de replanteo y armado de acero sin contratiempos.", "Condiciones climaticas favorables, rinde de cuadrilla al 100%.", "Fundicion de concreto con ensayos de cilindros conformes."]
log_civ_warn = ["Aparicion de humedad en el fondo de la excavacion.", "Lluvias aisladas retrasan levemente el armado de formaleta.", "Rendimiento de maquinaria disminuye por saturacion del suelo.", "Se solicita equipo de bombeo preventivo."]
log_civ_crit = ["Nivel freatico alto impide continuar excavacion. Carcamos inundados.", "Paralizacion parcial. Se instalan bombas de achique 24/7.", "Rendimiento de cuadrilla cae al 30%. Terreno inestable y lodoso.", "Derrumbe de taludes en cimentacion mayor por saturacion."]

admin_civ_ok = ["Aprobacion de APU sin desvios.", "Suministro de concreto premezclado facturado segun presupuesto.", "Reporte de horas hombre dentro de linea base."]
admin_civ_crit = ["Sobrecosto grave: Aprobacion de adicionales por alquiler de bombas y mangueras.", "Impacto a flujo de caja por horas extras de cuadrillas en achique.", "Disputa con contratista por rendimientos no contemplados en contrato."]

log_eq_ok = ["Planos de taller aprobados por interventoria.", "Orden de compra emitida a fabrica.", "Kick-off meeting con fabricante realizado."]
log_eq_crit = ["Notificacion de fuerza mayor: Escasez de microchips retrasa celdas.", "Equipos terminados pero retenidos en puerto asiatico por falta de contenedores.", "Inspeccion FAT aplazada por problemas en aduana de origen."]
admin_eq_crit = ["Fluctuacion de la TRM encarece costos de nacionalizacion.", "Pago de multas por bodegaje en puerto local.", "Reclamacion comercial al fabricante por incumplimiento de cronograma de embarque."]

log_ing_ok = ["Levantamiento topografico entregado y firmado.", "Memoria de calculo civil aprobada para construccion.", "Reuniones de coordinacion interdisciplinaria efectivas.", "Planos electromecanicos sin observaciones mayores."]
admin_ing_ok = ["Actas de avance de consultoria liquidadas.", "Cierre de hitos de diseno conforme al flujo de caja.", "Sin alertas contractuales en la fase de ingenieria."]

# Despliegue Semanal
fechas = pd.date_range(start='2025-07-01', periods=60, freq='W')
weekly_records = []

for _, row in df_base.iterrows():
    for w in range(1, 61):
        if w < row['s_start']: pv_factor = 0.0
        elif w >= row['s_end']: pv_factor = 1.0
        else: pv_factor = ((w - row['s_start']) / (row['s_end'] - row['s_start'])) ** 1.2
        pv_w = round(row['BAC'] * pv_factor, 2)

        if w <= 20:
            if row['pv_20'] > 0:
                ev_w = round(row['ev_20'] * (pv_w / row['pv_20']), 2)
                ac_w = round(row['ac_20'] * (pv_w / row['pv_20']), 2)
            else: ev_w, ac_w = 0.0, 0.0
        else: ev_w, ac_w = np.nan, np.nan

        # --- MOTOR DE TEXTOS NLP ---
        c_o, c_a = "", ""
        if w <= 20 and pv_w > 0:
            cap = row['Capitulo']

            # Obras Civiles: Degradación gradual del sentimiento
            if cap.startswith('3.'):
                if w < 7:
                    c_o, c_a = random.choice(log_civ_ok), random.choice(admin_civ_ok)
                elif 7 <= w <= 10:
                    c_o, c_a = random.choice(log_civ_warn), random.choice(admin_civ_ok)
                else:
                    c_o, c_a = random.choice(log_civ_crit), random.choice(admin_civ_crit)

            # Equipos: Atraso logístico de impacto alto
            elif cap.startswith('4.'):
                if w < 12:
                    c_o, c_a = random.choice(log_eq_ok), random.choice(admin_ing_ok)
                else:
                    c_o, c_a = random.choice(log_eq_crit), random.choice(admin_eq_crit)

            # Ingeniería y Gestión: Saludables / Positivos
            elif cap.startswith('1.') or cap.startswith('2.'):
                c_o, c_a = random.choice(log_ing_ok), random.choice(admin_ing_ok)

        else:
            c_o, c_a = "-", "-"

        weekly_records.append({
            'Fecha': fechas[w-1].strftime('%Y-%m-%d'), 'Semana': w, 'WBS': row['WBS'],
            'Capitulo': row['Capitulo'], 'Subcapitulo': row['Subcapitulo'], 'Actividad': row['Actividad'],
            'PV': pv_w, 'EV': ev_w, 'AC': ac_w, 'Comentarios_Obra': clean_text(c_o), 'Comentarios_Admin': clean_text(c_a)
        })

df_export = pd.DataFrame(weekly_records)
df_export.to_csv('data_proyecto_SE.csv', index=False, sep=';', decimal=',', na_rep='')
print("Script 1 Finalizado. Generados 16,800 registros con textos NLP-Ready (sin acentos).")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as patches
from scipy.interpolate import make_interp_spline

# =============================================================================
# SCRIPT 2: PROCESAMIENTO ANALÍTICO Y DASHBOARD EJECUTIVO
# Consume 'data_proyecto_SE.csv'
# =============================================================================

# 1. Ingesta de Datos
try:
    df = pd.read_csv('data_proyecto_SE.csv', sep=';', decimal=',')
except FileNotFoundError:
    raise FileNotFoundError("Debe ejecutar primero el Script 1 para generar 'data_proyecto_SE.csv'.")

# 2. Segmentación de Cortes
df_w20 = df[df['Semana'] == 20].copy()
df_w60 = df[df['Semana'] == 60].copy()

def calcular_kpis(df_base):
    df_base['CPI'] = np.where(df_base['AC'] > 0, (df_base['EV'] / df_base['AC']), 1.0)
    df_base['SPI'] = np.where(df_base['PV'] > 0, (df_base['EV'] / df_base['PV']), 1.0)
    df_base['CV'] = df_base['EV'] - df_base['AC']
    df_base['SV'] = df_base['EV'] - df_base['PV']
    return df_base.round(2)

# =============================================================================
# EXPORTACIÓN DE TABLAS DE MEDIDAS PARA BI
# =============================================================================
bac_df = df_w60[['WBS', 'PV']].rename(columns={'PV': 'BAC'})

# Actividades
actividades = df_w20[['WBS', 'Capitulo', 'Subcapitulo', 'Actividad', 'PV', 'EV', 'AC']].copy()
actividades = actividades.merge(bac_df, on='WBS')
actividades = calcular_kpis(actividades)
actividades = actividades[['WBS', 'Capitulo', 'Subcapitulo', 'Actividad', 'BAC', 'PV', 'EV', 'AC', 'CPI', 'SPI', 'CV', 'SV']]
actividades.to_csv('kpis_por_actividad_pbi.csv', index=False, sep=';', decimal=',')

# Capítulos
capitulos = df_w20.groupby('Capitulo')[['PV', 'EV', 'AC']].sum().reset_index()
bac_cap = df_w60.groupby('Capitulo')['PV'].sum().reset_index().rename(columns={'PV': 'BAC'})
capitulos = capitulos.merge(bac_cap, on='Capitulo')
capitulos = calcular_kpis(capitulos)
capitulos.to_csv('kpis_por_capitulo_pbi.csv', index=False, sep=';', decimal=',')

# Macro
BAC_tot = bac_df['BAC'].sum()
PV_tot, EV_tot, AC_tot = df_w20['PV'].sum(), df_w20['EV'].sum(), df_w20['AC'].sum()
CPI_tot = EV_tot / AC_tot if AC_tot > 0 else 1.0
SPI_tot = EV_tot / PV_tot if PV_tot > 0 else 1.0
EAC_tot = BAC_tot / CPI_tot if CPI_tot > 0 else BAC_tot
ETC_tot = EAC_tot - AC_tot
VAC_tot = BAC_tot - EAC_tot
TCPI_tot = (BAC_tot - EV_tot) / (BAC_tot - AC_tot) if (BAC_tot - AC_tot) != 0 else 1.0
SAC, Retraso_Meses = 15.0, 15.0 - (15.0 * SPI_tot)

kpis_macro = pd.DataFrame({
    'Medida': ['BAC', 'PV_Mes5', 'EV_Mes5', 'AC_Mes5', 'CPI_Global', 'SPI_Global', 'EAC', 'ETC', 'VAC', 'SV', 'CV'],
    'Valor': [BAC_tot, PV_tot, EV_tot, AC_tot, CPI_tot, SPI_tot, EAC_tot, ETC_tot, VAC_tot, EV_tot - PV_tot, EV_tot - AC_tot]
}).round(2)
kpis_macro.to_csv('medidas_kpis_macro_pbi.csv', index=False, sep=';', decimal=',')

# =============================================================================
# RENDERIZADO DEL DASHBOARD EJECUTIVO (MATPLOTLIB)
# =============================================================================
c_bg, c_header_bg = '#D3D3D3', '#003366'
c_gauge_red, c_gauge_yellow, c_gauge_green = '#D32F2F', '#FBC02D', '#388E3C'
c_line_pv, c_line_ev, c_line_ac = '#003366', '#1976D2', '#8B0000'

fig = plt.figure(figsize=(20, 12))
fig.patch.set_facecolor(c_bg)
gs = gridspec.GridSpec(5, 4, height_ratios=[0.08, 0.25, 0.35, 0.25, 0.07], hspace=0.5, wspace=0.2, width_ratios=[1, 1, 1, 1])

# --- A. HEADER ---
ax_header = fig.add_subplot(gs[0, :])
ax_header.axis('off')
ax_header.add_patch(patches.Rectangle((0, 0), 1, 1, transform=ax_header.transAxes, color=c_header_bg))
ax_header.text(0.02, 0.5, "DASHBOARD DE CONTROL TÉCNICO: SUBESTACIÓN 115kV | PROYECTOS & DATOS", color='white', fontsize=18, fontweight='bold', va='center')

# --- B. GAUGES ---
def draw_gauge_centered(ax, value, title):
    ax.axis('off'); ax.set_aspect('equal')
    r, w, cy = 0.60, 0.08, 0.35

    ax.add_patch(patches.Wedge((0.5, cy), r, 99, 180, width=w, transform=ax.transAxes, color='#FFCDD2', zorder=1))
    ax.add_patch(patches.Wedge((0.5, cy), r, 99, 180, width=w, transform=ax.transAxes, color=c_gauge_red, alpha=0.8, zorder=2))
    ax.add_patch(patches.Wedge((0.5, cy), r, 90, 99, width=w, transform=ax.transAxes, color='#FFF9C4', zorder=1))
    ax.add_patch(patches.Wedge((0.5, cy), r, 90, 99, width=w, transform=ax.transAxes, color=c_gauge_yellow, alpha=0.8, zorder=2))
    ax.add_patch(patches.Wedge((0.5, cy), r, 0, 90, width=w, transform=ax.transAxes, color='#C8E6C9', zorder=1))
    ax.add_patch(patches.Wedge((0.5, cy), r, 0, 90, width=w, transform=ax.transAxes, color=c_gauge_green, alpha=0.8, zorder=2))

    ax.text(0.5 - r + w/2, cy - 0.05, "0", ha='center', transform=ax.transAxes, fontsize=12, fontweight='bold')
    ax.text(0.5 + r - w/2, cy - 0.05, "2", ha='center', transform=ax.transAxes, fontsize=12, fontweight='bold')

    color_val = c_gauge_red if value < 0.95 else (c_gauge_yellow if value < 1.0 else c_gauge_green)
    val_clamped = max(0.5, min(1.5, value))
    theta = np.radians(180 - (val_clamped - 0.5) * 180)
    x, y = 0.5 + (r - 0.02) * np.cos(theta), cy + (r - 0.02) * np.sin(theta)

    ax.plot([0.5, x], [cy, y], color='#212121', lw=5, transform=ax.transAxes, zorder=3)
    ax.add_patch(patches.Circle((0.5, cy), 0.05, color='#212121', transform=ax.transAxes, zorder=4))

    r_val = r + 0.07
    x_val, y_val = 0.5 + r_val * np.cos(theta), cy + r_val * np.sin(theta)
    ha_val = 'right' if np.degrees(theta) > 135 else ('left' if np.degrees(theta) < 45 else 'center')
    va_val = 'bottom' if ha_val == 'center' else 'center'

    ax.text(x_val, y_val, f"{value:.2f}", fontsize=24, fontweight='bold', color=color_val, ha=ha_val, va=va_val, transform=ax.transAxes)
    ax.text(0.5, cy - 0.28, title, fontsize=16, fontweight='bold', ha='center', transform=ax.transAxes)

draw_gauge_centered(fig.add_subplot(gs[1, 0]), CPI_tot, "CPI")
draw_gauge_centered(fig.add_subplot(gs[1, 1]), SPI_tot, "SPI")

# --- C. INDICADORES (KPIs) ---
ax_kpis = fig.add_subplot(gs[1, 2:])
ax_kpis.axis('off')
kpis_list = [("TCPI", f"{TCPI_tot:.2f}", 0.05, 0.55), ("EAC/BAC", f"{EAC_tot/BAC_tot:.2f}", 0.55, 0.55),
             ("SAC (Meses)", f"{SAC:.1f}", 0.05, 0.1), ("VAC (USD)", f"${VAC_tot:,.0f}", 0.55, 0.1)]

for title, val, x, y in kpis_list:
    ax_kpis.add_patch(patches.Rectangle((x, y), 0.4, 0.35, transform=ax_kpis.transAxes, color='#E0E0E0', ec='none'))
    ax_kpis.text(x+0.2, y+0.22, title, ha='center', fontsize=11, fontweight='bold')
    ax_kpis.text(x+0.2, y+0.08, val, ha='center', fontsize=16, fontweight='bold', color='#003366' if 'VAC' not in title else c_gauge_red)

# --- D. CURVA S (Serie de Tiempo Semanal) ---
ax_scurve = fig.add_subplot(gs[2, :2])
ax_scurve.set_facecolor('white')

# Preparación de datos agregados por semana
agrupado_semanal = df.groupby('Semana')[['PV', 'EV', 'AC']].sum().reset_index()
# El eje X lo manejaremos en meses para limpieza (Semanas / 4)
x_semanas = agrupado_semanal['Semana'].values / 4.0
y_pv = agrupado_semanal['PV'].values

# EV y AC solo existen hasta la semana 20
df_hasta_20 = agrupado_semanal[agrupado_semanal['Semana'] <= 20]
x_corte = df_hasta_20['Semana'].values / 4.0
y_ev = df_hasta_20['EV'].values
y_ac = df_hasta_20['AC'].values

ax_scurve.plot(x_semanas, y_pv, label='PV (Valor Planificado)', linestyle='--', color=c_line_pv, lw=2.5)
ax_scurve.plot(x_corte, y_ev, label='EV (Valor Ganado)', color=c_line_ev, lw=2.5)
ax_scurve.plot(x_corte, y_ac, label='AC (Costo Real)', color=c_line_ac, lw=2.5)

ax_scurve.axvline(x=5, color='gray', linestyle=':', alpha=0.7)
ax_scurve.text(5.2, BAC_tot*0.1, 'Corte: Mes 5', rotation=90, color='gray', fontweight='bold')

ax_scurve.set_title("CURVA S: DESEMPEÑO ACUMULADO AL MES 5", fontweight='bold')
ax_scurve.set_xticks(np.arange(0, 16, 1))
ax_scurve.set_xlim(0, 15)
ax_scurve.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"${x/1000000:.1f}M"))
ax_scurve.set_xlabel("Meses del Proyecto")
ax_scurve.grid(True, linestyle=':', alpha=0.6)
ax_scurve.legend(loc='upper left')

if len(x_corte) > 0:
    # Format values for labels
    ev_label = f"EV: ${y_ev[-1]/1000000:.1f}M"
    ac_label = f"AC: ${y_ac[-1]/1000000:.1f}M"

    # Use a fixed offset for vertical stacking to ensure no overlap and clear separation
    vertical_offset = 50000  # Adjust this value as needed for desired spacing
    ax_scurve.text(x_corte[-1] + 3, y_ac[-1] + vertical_offset, ac_label, color=c_line_ac, fontsize=10, fontweight='bold', ha='left', va='center')
    ax_scurve.text(x_corte[-1] + 3, y_ev[-1] - vertical_offset, ev_label, color=c_line_ev, fontsize=10, fontweight='bold', ha='left', va='center')

# --- E. MATRIZ CPI vs SPI ---
ax_mat = fig.add_subplot(gs[2, 2:])
ax_mat.set_facecolor('white')
ax_mat.axhline(1, color='black', lw=1); ax_mat.axvline(1, color='black', lw=1)
ax_mat.fill_between([0.5, 1], 1, 1.3, color='#FFF3E0', alpha=0.5)
ax_mat.fill_between([1, 1.3], 1, 1.3, color='#E8F5E9', alpha=0.8)
ax_mat.fill_between([0.5, 1], 0.7, 1, color='#FFEBEE', alpha=0.8)
ax_mat.fill_between([1, 1.3], 0.7, 1, color='#FFF3E0', alpha=0.8)

colores = ['#4CAF50', '#2196F3', '#FF9800', '#9C27B0', '#F44336', '#00BCD4', '#E91E63']

for i, row in capitulos.iterrows():
    ax_mat.scatter(row['SPI'], row['CPI'], s=row['BAC']/4000, color=colores[i], edgecolors='black', alpha=0.8)
    label = row['Capitulo'].split(' ', 1)[1] if ' ' in row['Capitulo'] else row['Capitulo']
    ax_mat.text(row['SPI'] + 0.01, row['CPI'], label, fontsize=8, ha='left', va='center', fontweight='bold', color='black')

ax_mat.set_xlim(0.65, 1.3); ax_mat.set_ylim(0.65, 1.3)
ax_mat.set_xlabel('SPI'); ax_mat.set_ylabel('CPI')
ax_mat.set_title("MATRIZ CPI vs SPI POR CAPÍTULO", fontweight='bold')
ax_mat.grid(True, linestyle=':', alpha=0.6)

# --- F. TABLAS (Valores y Prognosis) ---
TITLE_Y_GAP = 1.05
TABLE_FONT_SIZE = 10

ax_t1 = fig.add_subplot(gs[3, :2]); ax_t1.axis('off')
ax_t1.set_title("VALORES ACTUALES (MONEDA)", fontweight='bold', y=TITLE_Y_GAP)
t1 = ax_t1.table(
    cellText=[["Presupuesto (BAC)", f"${BAC_tot:,.0f}"], ["Valor Planificado (PV)", f"${PV_tot:,.0f}"],
              ["Valor Ganado (EV)", f"${EV_tot:,.0f}"], ["Costo Actual (AC)", f"${AC_tot:,.0f}"]],
    colLabels=["Métrica Actual", "Valor"], loc='upper center', cellLoc='center')
t1.auto_set_font_size(False); t1.set_fontsize(TABLE_FONT_SIZE); t1.scale(1, 1.4)

ax_t2 = fig.add_subplot(gs[3, 2:]); ax_t2.axis('off')
ax_t2.set_title("CÁLCULOS DE PROGNOSIS", fontweight='bold', y=TITLE_Y_GAP)
t2 = ax_t2.table(
    cellText=[["VAC (Variación al Finalizar)", f"${VAC_tot:,.0f}"], ["Retraso Estimado (Meses)", f"{Retraso_Meses:.1f}"],
              ["ETC (Estimación para Completar)", f"${ETC_tot:,.0f}"], ["CPI al Finalizar", f"{CPI_tot:.2f}"]],
    colLabels=["Métrica Pronóstico", "Valor"], loc='upper center', cellLoc='center')
t2.auto_set_font_size(False); t2.set_fontsize(TABLE_FONT_SIZE); t2.scale(1, 1.4)

# --- G. FOOTER INSTITUCIONAL ---
ax_foot = fig.add_subplot(gs[4, :])
ax_foot.axis('off')
ax_foot.add_patch(patches.Rectangle((0, 0), 1, 1, transform=ax_foot.transAxes, color=c_header_bg))
ax_foot.text(0.02, 0.5, "ING. EMILIO PALACÍN GÓMEZ", color='white', fontsize=14, fontweight='bold', va='center')
ax_foot.text(0.5, 0.5, "Consultor PMO | Project Manager PMP® | BI, Reporting Ejecutivo y Control de Proyectos", color='white', fontsize=11, va='center', ha='center')
ax_foot.text(0.98, 0.5, "www.linkedin.com/in/emiliopalacin", color='white', fontsize=11, va='center', ha='right')

fig.subplots_adjust(left=0.05, right=0.99, top=0.9, bottom=0.05, hspace=0.6, wspace=0.2)
plt.savefig('dashboard_proyecto_SE.png', dpi=300, bbox_inches='tight')
plt.show()

print("Script 2 Finalizado. Dashboard generado y exportado como 'dashboard_proyecto_SE.png'.")